# Démonstration d'entraînement U-TILISE

Ce notebook montre comment lancer un entraînement du modèle U-TILISE de manière interactive.

**Contenu :**
1. Chargement et fusion de la configuration
2. Préparation des datasets (train / val) avec un sous-ensemble réduit
3. Instanciation du modèle, de l'optimiseur et du scheduler
4. Lancement de l'entraînement (quelques epochs)
5. Suivi des courbes d'entraînement via TensorBoard

> **Prérequis :** l'environnement conda `cloud_reconstruction` doit être activé.
> Un fichier HDF5 doit être accessible (chemin configuré dans le YAML).

In [1]:
import os
import sys
import matplotlib.pyplot as plt
# Se placer à la racine du projet
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Répertoire de travail : {PROJECT_ROOT}")

# Charger les variables d'environnement depuis .env
from dotenv import load_dotenv
load_dotenv()


Répertoire de travail : /home/SPeillet/rpg_3STR/cloud_reconstruction/U-TILISE


True

## 1. Configuration

La configuration est obtenue par fusion de 3 niveaux :
- `configs/default.yaml` — paramètres par défaut
- `configs/config_run_train.yaml` — surcharges d'entraînement
- Modifications manuelles ci-dessous (subset, nombre d'epochs, etc.)

In [2]:
from omegaconf import OmegaConf
from src import config_utils, data_utils, utils

# Charger et fusionner les configs
cfg_default = config_utils.read_config("configs/default.yaml")
cfg_train = config_utils.read_config("configs/config_run_train.yaml")
config = OmegaConf.merge(cfg_default, cfg_train)

# --- Adaptations pour la démo ---
# Sous-ensemble réduit pour un entraînement rapide
config.data.subset = 10
# Nombre d'epochs limité
config.training_settings.num_epochs = 3
# Répertoire de sortie
SAVE_DIR = os.path.join(os.environ.get("OUTPUT_DIR", "/tmp"), "training_demo")
config.output.output_directory = SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

print("=== Configuration (extrait) ===")
print(f"  Dataset       : {config.data.dataset}")
print(f"  HDF5          : {config.data.hdf5_file}")
print(f"  Séquence max  : {config.data.max_seq_length}")
print(f"  SAR           : {config.data.use_sar}")
print(f"  Masquage      : {config.mask.mask_type}")
print(f"  Batch size    : {config.training_settings.batch_size}")
print(f"  Epochs        : {config.training_settings.num_epochs}")
print(f"  Subset        : {config.data.subset}")
print(f"  Sortie        : {SAVE_DIR}")


=== Configuration (extrait) ===
  Dataset       : circa
  HDF5          : /mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5
  Séquence max  : 5
  SAR           : mix_closest
  Masquage      : random_fully_masked
  Batch size    : 2
  Epochs        : 3
  Subset        : 10
  Sortie        : /mnt/DATA_10T/data_rpg/outputs/U-TILISE/training_demo


## 2. Préparation des datasets et dataloaders

In [3]:
import logging
import torch

from src.logger import prepare_logger

# Logger simple pour la démo
logger = prepare_logger("demo", level=logging.INFO, log_to_console=True)

# Seed pour la reproductibilité
utils.set_seed(42)

# Datasets train / val
train_dset = data_utils.get_dataset(config, phase="train", logger=logger)
val_dset = data_utils.get_dataset(config, phase="val", logger=logger)

print(f"\nTaille du train set : {len(train_dset)} patches")
print(f"Taille du val set   : {len(val_dset)} patches")
print(f"Canaux              : {train_dset.num_channels}")
print(f"Taille image        : {train_dset.image_size}")

Backend detecte : Fichier HDF5
   Fichier : /mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5
Backend detecte : Fichier HDF5
   Fichier : /mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5



Taille du train set : 3249 patches
Taille du val set   : 1188 patches
Canaux              : 14
Taille image        : (256, 256)


In [4]:
# Dataloaders
subset = config.data.get("subset", False)
generator = torch.Generator()
generator.manual_seed(42)

train_loader = data_utils.get_dataloader(
    train_dset, config, drop_last=True, shuffle=True,
    generator=generator, subset=subset,
)
val_loader = data_utils.get_dataloader(
    val_dset, config, drop_last=False, shuffle=True,
    generator=generator, subset=subset,
)

print(f"Batches train : {len(train_loader)}")
print(f"Batches val   : {len(val_loader)}")

Batches train : 5
Batches val   : 5


## 3. Visualisation d'un échantillon

Avant de lancer l'entraînement, visualisons un échantillon pour vérifier les données.

In [5]:
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display, clear_output
from src import visutils
from src.data_utils import extract_sample

# Charger un batch
batch = next(iter(train_loader))
inputs, target, masks, mask_valid, cloud_mask, indices_rgb, index_nir = extract_sample(batch)
idx_rgb = indices_rgb.int().tolist()

print(f"Entrée (x)   : {inputs.shape}  — (B, T, C, H, W)")
print(f"Cible (y)    : {target.shape}")
print(f"Masques      : {masks.shape}")

# --- Visualisation interactive avec slider temporel ---
def _draw_rows(images, ncols, t_offset, row_labels, axes, brightness_factor):
    images = visutils.apply_brightness_factor(images, factor=brightness_factor)
    for idx, ax in enumerate(axes.flat):
        ax.imshow(images[idx])
        ax.axis('off')
        if idx < ncols:
            ax.set_title(f't{t_offset + idx}', fontsize=11, fontweight='bold')
    for i, label in enumerate(row_labels):
        axes[i, 0].annotate(label, xy=(-0.12, 0.5), xycoords='axes fraction',
                            fontsize=10, fontweight='bold',
                            ha='right', va='center', rotation=90)

target_vis = target[0][:, idx_rgb].permute(0, 2, 3, 1).cpu().clone()
inp_vis = inputs[0][:, idx_rgb].permute(0, 2, 3, 1).cpu().clone()
masks_vis = masks[0][:, 0].cpu().clone()

T = target_vis.shape[0]
n_vis = min(8, T)

out = widgets.Output()
slider = widgets.IntSlider(
    value=0, min=0, max=max(T - n_vis, 0), step=1,
    description='t_start :', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%'),
)
info = widgets.Label(value=f'Série : {T} dates  |  Fenêtre visible : {n_vis}')

def _update(_change=None):
    t0 = slider.value
    s = slice(t0, t0 + n_vis)
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(nrows=3, ncols=n_vis, figsize=(2.2 * n_vis, 7.5))
        if n_vis == 1:
            axes = axes.reshape(3, 1)
        imgs = torch.cat([target_vis[s], inp_vis[s]], dim=0)
        _draw_rows(imgs, n_vis, t0, ['Cible', 'Entrée masquée'], axes[:2], 3.0)
        for j in range(n_vis):
            axes[2, j].imshow(masks_vis[t0 + j], cmap='gray', vmin=0, vmax=1)
            axes[2, j].axis('off')
        axes[2, 0].annotate('Masque', xy=(-0.12, 0.5), xycoords='axes fraction',
                            fontsize=10, fontweight='bold',
                            ha='right', va='center', rotation=90)
        plt.subplots_adjust(left=0.10, wspace=0.05, hspace=0.12)
        plt.show()

slider.observe(_update, names='value')
display(widgets.VBox([info, slider, out]))
_update()

Entrée (x)   : torch.Size([2, 5, 14, 256, 256])  — (B, T, C, H, W)
Cible (y)    : torch.Size([2, 5, 10, 256, 256])
Masques      : torch.Size([2, 5, 1, 256, 256])


## 4. Instanciation du modèle

In [6]:
# Créer le répertoire expérience et sauvegarder la config
config.output.experiment_folder = utils.create_output_directory(config)
config.output.checkpoint_dir = os.path.join(config.output.experiment_folder, "checkpoints")
os.makedirs(config.output.checkpoint_dir, exist_ok=True)
config_utils.write_config(config, os.path.join(config.output.experiment_folder, "config.yaml"))

# Modèle
input_dim = train_dset.num_channels
model, args_model = utils.get_model(config, input_dim, logger)
n_params = utils.count_model_parameters(model)
print(f"\nModèle       : U-TILISE")
print(f"Entrée       : {input_dim} canaux")
print(f"Paramètres   : {n_params:,}")

# Optimiseur et scheduler
optimizer = utils.get_optimizer(config, model, logger)
scheduler = utils.get_scheduler(config, optimizer, logger)


Modèle       : U-TILISE
Entrée       : 14 canaux
Paramètres   : 1,355,978


## 5. Entraînement

On utilise le `Trainer` intégré qui gère :
- La boucle train / val par epoch
- L'écriture des logs TensorBoard dans `{experiment_folder}/tb/`
- La sauvegarde des meilleurs poids

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

trainer = utils.get_trainer(
    config, train_dset, val_dset,
    train_loader, val_loader,
    model, optimizer, scheduler, device,
)

# Lancer l'entraînement
trainer.train()


Training from scratch.



Device : cuda:0


val:	Epoch: 0	l1_loss: 0.37589	total_loss: 0.37589	mae: 0.27978	mse: 0.11471	rmse: 0.33813	ssim: 0.19477	psnr: 9.43255	sam: 0.30886	

Start training...

Epoch: 100%|██████████| 3/3 [00:24<00:00,  8.32s/it, best_validation_loss=0.229, epoch=2, training_loss=nan, validation_loss=0.226]


Training finished!
Training time: 0d 0h 0m 24s

Best model at epoch: 2
Validation loss of the best model: 0.2263


## 6. Suivi via TensorBoard

Les courbes de loss et métriques sont enregistrées automatiquement dans le sous-dossier `tb/`.
On peut les visualiser directement dans le notebook :

In [8]:
tb_log_dir = os.path.join(config.output.experiment_folder, "tb")
print(f"Répertoire TensorBoard : {tb_log_dir}")

%load_ext tensorboard
%tensorboard --logdir {tb_log_dir}

Répertoire TensorBoard : /mnt/DATA_10T/data_rpg/outputs/U-TILISE/training_demo/2026-04-14_17-36/tb


Launching TensorBoard...

## 7. Résumé

Les fichiers générés se trouvent dans le répertoire d'expérience :

In [9]:
print(f"Répertoire expérience : {config.output.experiment_folder}\n")
for root, dirs, files in os.walk(config.output.experiment_folder):
    level = root.replace(config.output.experiment_folder, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

Répertoire expérience : /mnt/DATA_10T/data_rpg/outputs/U-TILISE/training_demo/2026-04-14_17-36

2026-04-14_17-36/
  training.log
  config.yaml
  tb/
    events.out.tfevents.1776181012.LNV2309W016.1749808.0
  checkpoints/
    Model_last.pth
    Model_best.pth


---

**Pour un entraînement complet**, utiliser le script en ligne de commande :

```bash
python run_train.py configs/config_run_train.yaml --save_dir /chemin/vers/sortie/
```

Voir le README pour les configs détaillées (architecture v4, SAR asc+desc, etc.).